In [7]:
# %% Cell 0: Profiling (optional) and Imports

import os
import numpy as np
import matplotlib.pyplot as plt
import sys
import logging
from typing import Optional

# --- Add Project Root to sys.path ---
NOTEBOOK_DIR = os.path.abspath('')
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR) 
PROJECT_ROOT = os.path.join(PROJECT_ROOT, 'm_detector_python')
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print(f"Added project root to sys.path: {PROJECT_ROOT}")



# --- Import Custom Modules ---
from src.config_loader import MDetectorConfigAccessor 
# --- REMOVED: Obsolete imports from old validation utility files ---
# from src.utils.validation_utils import (...)
from src.visualization.metrics_plots import plot_roc_curve, plot_precision_recall_curve
from src.core.constants import OcclusionResult

print("Cell 0: Imports and Paths - OK")

Cell 0: Imports and Paths - OK


In [8]:
# %% Cell 1: Configuration and Path Setup

# --- Load Configuration using Accessor ---
config_accessor: Optional[MDetectorConfigAccessor] = None
try:
    # config_file_path = os.path.join(PROJECT_ROOT, 'config/m_detector_config.yaml') # Base config for defaults
    config_file_path = os.path.join(PROJECT_ROOT, 'config/config_best_trial.yaml') # Base config for defaults
    
    config_accessor = MDetectorConfigAccessor(config_file_path)
    print(f"Base configuration loaded successfully using MDetectorConfigAccessor from: {config_file_path}")
except FileNotFoundError:
    print(f"ERROR: Base config file not found at {config_file_path}. Cannot load default paths.")
    config_accessor = None # Ensure it's None if loading fails
except Exception as e:
    print(f"Error loading base config with MDetectorConfigAccessor: {e}. Cannot load default paths.")
    config_accessor = None # Ensure it's None

default_gt_labels_dir = None
default_base_tuning_results_dir = None

if config_accessor:
    nuscenes_params = config_accessor.get_nuscenes_params()
    default_gt_labels_dir = nuscenes_params.get('label_path')
    if default_gt_labels_dir and not os.path.isabs(default_gt_labels_dir):
        default_gt_labels_dir = os.path.join(PROJECT_ROOT, default_gt_labels_dir)

    mdet_output_paths = config_accessor.get_mdetector_output_paths()
    default_base_tuning_results_dir = mdet_output_paths.get('save_path')
    if default_base_tuning_results_dir and not os.path.isabs(default_base_tuning_results_dir):
        default_base_tuning_results_dir = os.path.join(PROJECT_ROOT, default_base_tuning_results_dir)

GT_LABELS_BASE_DIR = default_gt_labels_dir if default_gt_labels_dir else "/path/to/your/nuscenes_labels_interpolated"
# BASE_TUNING_RESULTS_DIR = "/home/drugge/staff-umbrella/TeamHolgerResearch/drugge/m_detector_python/output/testGPU_RAY_SMALL_TEST_09_06_25"
BASE_TUNING_RESULTS_DIR = default_base_tuning_results_dir

print(f"\nUsing Base Directory for Tuning Results: {BASE_TUNING_RESULTS_DIR}")
print(f"Using GT Label Files From:               {GT_LABELS_BASE_DIR}")

if not os.path.isdir(BASE_TUNING_RESULTS_DIR):
    print(f"ERROR: Base tuning results directory does not exist: {BASE_TUNING_RESULTS_DIR}")
if not os.path.isdir(GT_LABELS_BASE_DIR):
    print(f"WARNING: GT labels directory does not exist: {GT_LABELS_BASE_DIR}")

# --- Discover Tuning Subdirectories ---
tuning_experiments_to_evaluate = []
if os.path.isdir(BASE_TUNING_RESULTS_DIR):
    for item_name in sorted(os.listdir(BASE_TUNING_RESULTS_DIR)): # Sort for consistent order
        item_path = os.path.join(BASE_TUNING_RESULTS_DIR, item_name)
        expected_config_file = os.path.join(item_path, f"config_tuned_{item_name}.yaml")
        if os.path.isdir(item_path) and os.path.exists(expected_config_file):
            tuning_experiments_to_evaluate.append({
                "name": item_name,
                "path": item_path,
                "config_path": expected_config_file
            })


if not tuning_experiments_to_evaluate:
    print(f"WARNING: No valid tuning subdirectories found in {BASE_TUNING_RESULTS_DIR}")
else:
    print(f"\nFound {len(tuning_experiments_to_evaluate)} tuning experiments to evaluate:")
    for exp in tuning_experiments_to_evaluate:
        print(f"  - Name: {exp['name']}, Path: {exp['path']}")


# --- Base Evaluation Parameters ---
base_eval_params = {
    "mdet_label_field_name": "mdet_label",
    "mdet_dynamic_label_value": OcclusionResult.OCCLUDING_IMAGE.value,
    "coordinate_tolerance_for_verification": 1e-3, # Small value for exact point matching
    "mdet_min_point_range_meters": 2.5, # Default, will be overridden by MDet config if available
    "mdet_max_point_range_meters": 50.0, # Default, will be overridden by MDet config if available
    "evaluate_only_keyframes": False  
}
if config_accessor: # Use base config for these general eval params
    point_filter_params = config_accessor.get_point_pre_filtering_params()
    base_eval_params["mdet_min_point_range_meters"] = point_filter_params['min_range_meters']
    base_eval_params["mdet_max_point_range_meters"] = point_filter_params['max_range_meters']

print(f"\nBase Evaluation Parameters:")
for key, value in base_eval_params.items():
    print(f"  {key}: {value}")

from nuscenes.nuscenes import NuScenes
nusc: Optional[NuScenes] = None
if config_accessor:
    nuscenes_params = config_accessor.get_nuscenes_params()
    try:
        print("\nLoading NuScenes instance for metrics... (This may take a moment)")
        nusc = NuScenes(version=nuscenes_params['version'], dataroot=nuscenes_params['dataroot'], verbose=False)
        print("NuScenes instance loaded successfully.")
    except Exception as e:
        print(f"ERROR: Failed to load NuScenes instance. Metrics calculation will likely fail. Error: {e}")
else:
    print("WARNING: Cannot load NuScenes instance because base config failed to load.")

print("\nCell 1: Configuration - OK")

Base configuration loaded successfully using MDetectorConfigAccessor from: /home/drugge/Unsupervised-Moving-Point-Detection/m_detector_python/config/config_best_trial.yaml

Using Base Directory for Tuning Results: /home/drugge/Unsupervised-Moving-Point-Detection/m_detector_python/output/iou_check
Using GT Label Files From:               /home/drugge/staff-umbrella/TeamHolgerResearch/drugge/nuscenes_labels_interpolated

Found 1 tuning experiments to evaluate:
  - Name: static_validation_run, Path: /home/drugge/Unsupervised-Moving-Point-Detection/m_detector_python/output/iou_check/static_validation_run

Base Evaluation Parameters:
  mdet_label_field_name: mdet_label
  mdet_dynamic_label_value: 0
  coordinate_tolerance_for_verification: 0.001
  mdet_min_point_range_meters: 2.5
  mdet_max_point_range_meters: 50.0
  evaluate_only_keyframes: False

Loading NuScenes instance for metrics... (This may take a moment)
NuScenes instance loaded successfully.

Cell 1: Configuration - OK


In [9]:
# %% Cell 2: Calculate Summary Metrics and Display Results

import pandas as pd
import multiprocessing
from tqdm import tqdm
import sys
import os
import torch
import re

try:
    multiprocessing.set_start_method('spawn', force=True)
    print("Multiprocessing start method set to 'spawn'.")
except RuntimeError:
    print("Multiprocessing start method already set.")

# This import is correct and points to our newly refactored file
from src.data_utils.validation_utils_torch import scene_metrics_worker


# --- Setup Evaluation Parameters ---

# OPTION 1: Evaluate ALL sweeps
eval_params_all_sweeps = base_eval_params.copy()
eval_params_all_sweeps["gt_velocity_threshold"] = 1.0 

# OPTION 2: Evaluate ONLY KEYFRAMES
eval_params_keyframe_only = base_eval_params.copy()
eval_params_keyframe_only["gt_velocity_threshold"] = 1.0 
eval_params_keyframe_only["evaluate_only_keyframes"] = True

# --- Choose which eval_params to use for the current run ---
# Change this line to switch between evaluation modes
current_eval_params_to_use = eval_params_all_sweeps
# current_eval_params_to_use = eval_params_keyframe_only

# --- Main Execution Logic ---
all_scene_summaries_from_pool = []
summary_df = pd.DataFrame()

if 'tuning_experiments_to_evaluate' not in locals() or not tuning_experiments_to_evaluate:
    print("ERROR: 'tuning_experiments_to_evaluate' not found or empty. Please run Cell 1 first.")
else:
    processing_cfg = config_accessor.get_processing_settings()
    output_cfg = config_accessor.get_mdetector_output_paths()
    max_cpu_workers = output_cfg.get('max_parallel_scenes', 16)
    available_gpu_ids = []
    if processing_cfg.get('device') == 'cuda' and torch.cuda.is_available():
        configured_gpu_ids = processing_cfg.get('gpu_ids', [])
        if configured_gpu_ids:
            available_gpu_ids = configured_gpu_ids
        else:
            available_gpu_ids = list(range(torch.cuda.device_count()))
    
    print(f"Resource Config: Max CPU workers = {max_cpu_workers}, Usable GPUs = {available_gpu_ids}")
    
    tasks_for_scene_pool = []
    for exp_info in tuning_experiments_to_evaluate:
        tuning_name = exp_info["name"]
        mdet_dir = exp_info["path"]
        config_path = exp_info["config_path"]
        
        scene_files = [f for f in os.listdir(mdet_dir) if f.startswith("mdet_results_") and f.endswith(".pt")]
        
        for mdet_scene_file in scene_files:
            tasks_for_scene_pool.append((
                tuning_name,
                os.path.join(mdet_dir, mdet_scene_file),
                GT_LABELS_BASE_DIR, # Use the renamed variable
                current_eval_params_to_use,
                PROJECT_ROOT,
                config_path,
                nusc
            ))

    if not tasks_for_scene_pool:
        print("No scene result (.pt) files found to process.")
    else:
        num_workers = min(max_cpu_workers, len(tasks_for_scene_pool))
        
        tasks_with_ids = []
        for i, task in enumerate(tasks_for_scene_pool):
            gpu_id_for_worker = available_gpu_ids[i % len(available_gpu_ids)] if available_gpu_ids else None
            tasks_with_ids.append((i, gpu_id_for_worker, *task))

        print(f"\nCalculating metrics in parallel for {len(tasks_with_ids)} scenes using {num_workers} CPU workers.")
        if available_gpu_ids:
            print(f"Work will be distributed across GPUs: {available_gpu_ids}")
        else:
            print("No GPUs specified or available. All work will be on the CPU.")

        with multiprocessing.Pool(processes=num_workers) as pool:
            with tqdm(total=len(tasks_with_ids), desc="Calculating Scene Metrics (PyTorch)") as pbar:
                for result in pool.imap_unordered(scene_metrics_worker, tasks_with_ids):
                    all_scene_summaries_from_pool.append(result)
                    pbar.update(1)
                    if "error" in result:
                        print(f"\n\n[!!!] CRITICAL ERROR in worker for {result.get('tuning_name')}/{result.get('scene_name', 'Unknown Scene')}:")
                        print(f"      Reason: {result['error']}")
                        print("      Terminating pool to prevent further processing with a flawed setup.")
                        pool.terminate()
                        break

    # --- Aggregate results from scenes back into experiments ---
    experiment_final_metrics = {}
    for scene_result in all_scene_summaries_from_pool:
        tuning_name = scene_result.get("tuning_name")
        if not tuning_name: continue

        if tuning_name not in experiment_final_metrics:
            experiment_final_metrics[tuning_name] = {'tp': 0, 'fp': 0, 'fn': 0, 'tn': 0, 'scenes_processed': 0, 'scenes_with_errors': []}

        if "error" in scene_result:
            experiment_final_metrics[tuning_name]['scenes_with_errors'].append(scene_result)
        else:
            experiment_final_metrics[tuning_name]['tp'] += scene_result.get('tp', 0)
            experiment_final_metrics[tuning_name]['fp'] += scene_result.get('fp', 0)
            experiment_final_metrics[tuning_name]['fn'] += scene_result.get('fn', 0)
            experiment_final_metrics[tuning_name]['tn'] += scene_result.get('tn', 0)
            experiment_final_metrics[tuning_name]['scenes_processed'] += 1


Multiprocessing start method set to 'spawn'.
Resource Config: Max CPU workers = 26, Usable GPUs = [0, 6]

Calculating metrics in parallel for 1 scenes using 1 CPU workers.
Work will be distributed across GPUs: [0, 6]


Calculating Scene Metrics (PyTorch): 100%|████████| 1/1 [00:27<00:00, 27.18s/it]

[DEBUG] Slicing with: skip=0, max=-1, init_sweeps=False


In [10]:
print("\n[DEBUG] Notebook is using these eval_params:")
import json
print(json.dumps(current_eval_params_to_use, indent=2))


[DEBUG] Notebook is using these eval_params:
{
  "mdet_label_field_name": "mdet_label",
  "mdet_dynamic_label_value": 0,
  "coordinate_tolerance_for_verification": 0.001,
  "mdet_min_point_range_meters": 2.5,
  "mdet_max_point_range_meters": 50.0,
  "evaluate_only_keyframes": false,
  "gt_velocity_threshold": 1.0
}


In [12]:
# %% Cell 3: Final Metrics Calculation and Display

import pandas as pd
import warnings

# Suppress divide-by-zero warnings for this cell, as we handle it in the code
warnings.filterwarnings('ignore', category=RuntimeWarning)

# This list will hold the data for our final DataFrame
results_list_for_df = []

# Loop through each experiment's aggregated results
for tuning_name, metrics in experiment_final_metrics.items():
    tp = metrics.get('tp', 0)
    fp = metrics.get('fp', 0)
    fn = metrics.get('fn', 0)
    tn = metrics.get('tn', 0)
    
    # --- Calculate Standard Metrics ---
    # Precision: Of all the points we PREDICTED as dynamic, how many were actually dynamic?
    # High precision -> Low false positives
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    
    # Recall (Sensitivity): Of all the points that were ACTUALLY dynamic, how many did we find?
    # High recall -> Low false negatives
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    
    # F1-Score: The harmonic mean of precision and recall. A good overall measure.
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    # Accuracy: Overall, what fraction of points did we classify correctly?
    # (Often less useful for imbalanced datasets, but good to have)
    total_points = tp + fp + fn + tn
    accuracy = (tp + tn) / total_points if total_points > 0 else 0.0
    
    # Intersection over Union (IoU) for the "dynamic" class:
    # A very common metric in detection tasks.
    iou_dynamic = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0

    # Append the calculated metrics for this experiment to our list
    results_list_for_df.append({
        'Tuning Name': tuning_name,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1_score,
        'Accuracy': accuracy,
        'IoU (Dynamic)': iou_dynamic,
        'TP': tp,
        'FP': fp,
        'FN': fn,
        'TN': tn,
        'Scenes Processed': metrics.get('scenes_processed', 0),
        'Errors': len(metrics.get('scenes_with_errors', []))
    })

# --- Create and Display the DataFrame ---
if not results_list_for_df:
    print("No results were processed to create a summary DataFrame.")
else:
    # Create the DataFrame from our list of dictionaries
    summary_df = pd.DataFrame(results_list_for_df)
    
    # Sort the DataFrame by F1-Score in descending order to see the best performers on top
    summary_df = summary_df.sort_values(by='F1-Score', ascending=False).reset_index(drop=True)

    # --- Display the formatted DataFrame ---
    # Use Pandas styling for better readability
    display(summary_df.style.format({
        'Precision': '{:.4f}',
        'Recall': '{:.4f}',
        'F1-Score': '{:.4f}',
        'Accuracy': '{:.4f}',
        'IoU (Dynamic)': '{:.4f}',
        'TP': '{:,}',
        'FP': '{:,}',
        'FN': '{:,}',
        'TN': '{:,}',
        'Scenes Processed': '{:,}',
        'Errors': '{:,}'
    }).background_gradient(
        cmap='viridis', subset=['Precision', 'Recall', 'F1-Score', 'IoU (Dynamic)']
    ).bar(
        subset=['Scenes Processed'], color='#d65f5f'
    ).set_caption(
        "M-Detector Tuning Experiment Results"
    ).set_properties(**{'text-align': 'left'}))

# Restore default warning behavior
warnings.resetwarnings()

print("\nCell 3: Final Metrics Calculation - OK")

,Tuning Name,Precision,Recall,F1-Score,Accuracy,IoU (Dynamic),TP,FP,FN,TN,Scenes Processed,Errors
0,static_validation_run,0.4699,0.5870,0.5220,0.9844,0.3531,"82,831","93,452","58,274","9,462,938",1,0



Cell 3: Final Metrics Calculation - OK


In [3]:
# %% Cell 2: Calculate Summary Metrics and Display Results (PyTorch Accelerated)

import pandas as pd
import multiprocessing
from tqdm import tqdm
import sys
import os
import torch # --- NEW: Import torch ---

# --- MODIFIED: Import from the new PyTorch utility file ---
from src.utils.validation_utils_torch import calculate_metrics_for_experiment_hdf5_torch

# --- MODIFIED: Top-level worker function for metrics calculation ---
def metrics_worker(args_tuple):
    """
    Worker function to calculate metrics for a single tuning experiment using PyTorch.
    """
    # --- MODIFIED: Unpack worker_id and determine device ---
    worker_id, tuning_name, mdet_results_dir, gt_labels_dir, eval_params, project_root_for_worker, experiment_config_path, nusc_obj = args_tuple
    
    if torch.cuda.is_available():
        num_gpus = torch.cuda.device_count()
        device = torch.device(f"cuda:{worker_id % num_gpus}")
    else:
        device = torch.device("cpu")

    if project_root_for_worker not in sys.path:
         sys.path.append(project_root_for_worker)

    # --- MODIFIED: Call the new PyTorch-based function ---
    current_summary = calculate_metrics_for_experiment_hdf5_torch(
        mdet_experiment_dir=mdet_results_dir,
        gt_labels_base_dir=gt_labels_dir,
        eval_params=eval_params,
        mdet_experiment_config_path=experiment_config_path,
        nusc=nusc_obj,
        device=device # Pass the assigned device
    )
    if current_summary:
        current_summary['tuning_name'] = tuning_name
    return current_summary

# --- Setup Evaluation Parameters ---
# This section remains the same. It prepares the parameters for the run.

# OPTION 1: Evaluate ALL sweeps
eval_params_all_sweeps = base_eval_params.copy()
eval_params_all_sweeps["gt_velocity_threshold"] = 1.0 

# OPTION 2: Evaluate ONLY KEYFRAMES
eval_params_keyframe_only = base_eval_params.copy()
eval_params_keyframe_only["gt_velocity_threshold"] = 1.0 
eval_params_keyframe_only["evaluate_only_keyframes"] = True

# --- Choose which eval_params to use for the current run ---
# Change this line to switch between evaluation modes
current_eval_params_to_use = eval_params_all_sweeps
# current_eval_params_to_use = eval_params_keyframe_only


# --- Main Execution Logic ---
all_experiments_summaries_from_pool = []
summary_df = pd.DataFrame()

if 'tuning_experiments_to_evaluate' not in locals() or not tuning_experiments_to_evaluate:
    print("ERROR: 'tuning_experiments_to_evaluate' not found or empty. Please run Cell 1 first.")
else:
    tasks_for_metrics_pool = []
    # --- MODIFIED: Add worker_id to the task arguments using enumerate ---
    for worker_idx, experiment_info in enumerate(tuning_experiments_to_evaluate):
        tasks_for_metrics_pool.append((
            worker_idx, # Pass the worker's index for device assignment
            experiment_info["name"],
            experiment_info["path"],
            GT_LABELS_BASE_DIR_HDF5,
            current_eval_params_to_use,
            PROJECT_ROOT,
            experiment_info["config_path"],
            nusc 
        ))

    # --- (Parallel execution logic remains the same, but now it's much faster!) ---
    num_metrics_workers = min(torch.cuda.device_count() if torch.cuda.is_available() else multiprocessing.cpu_count(), len(tasks_for_metrics_pool))
    print(f"\nCalculating metrics in parallel for {len(tasks_for_metrics_pool)} tunings using {num_metrics_workers} workers...")
    with multiprocessing.Pool(processes=num_metrics_workers) as pool:
        with tqdm(total=len(tasks_for_metrics_pool), desc="Calculating All Tuning Metrics (PyTorch)") as pbar:
            for result_summary in pool.imap_unordered(metrics_worker, tasks_for_metrics_pool):
                if result_summary:
                    all_experiments_summaries_from_pool.append(result_summary)
                pbar.update(1)


    # --- NEW, ROBUST POST-PROCESSING AND ERROR DISPLAY ---
    successful_summaries = []
    failed_experiments_report = []

    for summary in all_experiments_summaries_from_pool:
        # A summary is considered a failure if it has an overall 'error' key
        # or if it has a list of 'scenes_with_errors'.
        if summary.get("error") or (summary.get("scenes_with_errors") and len(summary["scenes_with_errors"]) > 0):
            failed_experiments_report.append({
                "tuning_name": summary.get("tuning_name", "Unknown Tuning"),
                "overall_error": summary.get("error"),
                "scene_errors": summary.get("scenes_with_errors", [])
            })
        else:
            # If no errors are found, it's a success.
            successful_summaries.append(summary)

    # --- 1. Print the Detailed Failure Report ---
    if failed_experiments_report:
        print("\n\n" + "="*25 + " DETAILED FAILURE REPORT " + "="*25)
        for failure in failed_experiments_report:
            print(f"\n[Tuning]: {failure['tuning_name']}")
            if failure['overall_error']:
                print(f"  [Overall Experiment Error]: {failure['overall_error']}")
            if failure['scene_errors']:
                print("  [Failed Scenes]:")
                for scene_error in failure['scene_errors']:
                    print(f"    - Scene: {scene_error.get('scene_name', 'N/A')}")
                    # Use str() to handle potential non-string error objects gracefully
                    print(f"      Reason: {str(scene_error.get('error', 'No error message provided.'))}")
        print("\n" + "="*75)
    else:
        print("\n\n--- All experiments processed without returning errors. ---")

    # --- 2. Display the Comparative Table for ONLY the SUCCESSFUL runs ---
    if successful_summaries:
        metrics_to_display = [
            'tuning_name', 'Precision', 'Recall', 'F1',
            'overall_iou_dynamic', 'Accuracy', 'TP', 'FP', 'FN', 'TN',
            'num_scenes_successfully_evaluated', 'num_scenes_total_in_dir'
        ]
        data_for_df = []
        for summary in successful_summaries:
            row = {metric: summary.get(metric) for metric in metrics_to_display}
            data_for_df.append(row)

        summary_df = pd.DataFrame(data_for_df)
        if not summary_df.empty:
            summary_df = summary_df.set_index('tuning_name')
            eval_mode_description = " (Keyframes Only)" if current_eval_params_to_use.get("evaluate_only_keyframes") else " (All Sweeps)"
            print(f"\n\n===== Overall Comparative Metrics Summary (Successful Tunings){eval_mode_description} =====")
            float_cols = ['Precision', 'Recall', 'F1', 'iou', 'Accuracy']
            for col in float_cols:
                if col in summary_df.columns:
                    summary_df[col] = summary_df[col].apply(lambda x: f'{x:.4f}' if pd.notnull(x) else 'NaN')
            print(summary_df.to_string(columns=[col for col in metrics_to_display if col != 'tuning_name']))
        else:
            print("No successful data to display in summary DataFrame.")
    else:
        print("\nNo successful summaries were collected from any tuning experiment.")

print("\nCell 2: Summary Metrics - OK")


Calculating metrics in parallel for 2 tunings using 2 workers...


Calculating All Tuning Metrics (PyTorch): 100%|██████████| 2/2 [06:26<00:00, 193.24s/it]



========================= DETAILED FAILURE REPORT =========================

[Tuning]: AdapMCCB_Single_dmax_0p8
  [Failed Scenes]:
    - Scene: mdet_results_AdapMCCB_Single_dmax_0p8_scene_0.h5
      Reason: Point count mismatch. GT_filtered: 9544555, MDet: 9318071
    - Scene: mdet_results_AdapMCCB_Single_dmax_0p8_scene_1.h5
      Reason: Point count mismatch. GT_filtered: 9697495, MDet: 9519086
    - Scene: mdet_results_AdapMCCB_Single_dmax_0p8_scene_2.h5
      Reason: Point count mismatch. GT_filtered: 8936790, MDet: 8735512
    - Scene: mdet_results_AdapMCCB_Single_dmax_0p8_scene_3.h5
      Reason: Point count mismatch. GT_filtered: 9993315, MDet: 9762314
    - Scene: mdet_results_AdapMCCB_Single_dmax_0p8_scene_4.h5
      Reason: Point count mismatch. GT_filtered: 9450298, MDet: 9232545
    - Scene: mdet_results_AdapMCCB_Single_dmax_0p8_scene_5.h5
      Reason: Point count mismatch. GT_filtered: 10198073, MDet: 9965281
    - Scene: mdet_results_AdapMCCB_Single_dmax_0p8_scene_6.h5


In [4]:
# Select the tuning with the highest IoU
if not summary_df.empty and 'overall_iou_dynamic' in summary_df.columns:
    # Convert the formatted string back to float for comparison
    summary_df_numeric = summary_df.copy()
    summary_df_numeric['overall_iou_dynamic'] = pd.to_numeric(
        summary_df_numeric['overall_iou_dynamic'], errors='coerce'
    )
    
    # Find the tuning with maximum IoU
    best_tuning_idx = summary_df_numeric['overall_iou_dynamic'].idxmax()
    best_iou_value = summary_df_numeric.loc[best_tuning_idx, 'overall_iou_dynamic']
    
    print(f"\n===== BEST PERFORMING TUNING (Highest IoU) =====")
    print(f"Tuning Name: {best_tuning_idx}")
    print(f"IoU Score: {best_iou_value:.4f}")
    print("\nFull metrics for best tuning:")
    print(summary_df.loc[best_tuning_idx].to_string())
    
    # Optional: Get the full experiment info for the best tuning
    best_experiment_info = None
    for exp_info in tuning_experiments_to_evaluate:
        if exp_info["name"] == best_tuning_idx:
            best_experiment_info = exp_info
            break
    
    if best_experiment_info:
        print(f"\nBest tuning path: {best_experiment_info['path']}")
    
else:
    print("Cannot find best IoU tuning - summary_df is empty or missing IoU column")

Cannot find best IoU tuning - summary_df is empty or missing IoU column


In [ ]:
# %% Cell 2 (beginning - ensure this worker is defined before use in the pool)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import multiprocessing
from tqdm import tqdm
import os # For os.path.join if needed inside worker, though paths are passed
import sys # For sys.path manipulation in worker

# --- Top-level worker function for generating plot data points ---
def plot_data_worker(args_tuple):
    """
    Worker function to calculate metrics for a single tuning experiment
    at a specific GT velocity threshold.
    """
    tuning_name, mdet_results_dir, gt_labels_dir, eval_params_for_this_point, project_root_for_worker = args_tuple
    
    # Ensure src modules can be imported if worker runs in a new process context
    if project_root_for_worker not in sys.path:
         sys.path.append(project_root_for_worker)
    
    # Re-importing within the worker can sometimes be safer with multiprocessing
    from src.utils.validation_utils import calculate_metrics_for_experiment_hdf5

    experiment_summary = calculate_metrics_for_experiment_hdf5(
        mdet_experiment_dir=mdet_results_dir,
        gt_labels_base_dir=gt_labels_dir,
        eval_params=eval_params_for_this_point # This contains the specific threshold
    )
    
    threshold_used = eval_params_for_this_point.get("gt_velocity_threshold")
    
    # Return identifiers along with the result
    return tuning_name, threshold_used, experiment_summary

# --- The rest of your Cell 2 ---
# Ensure tuning_experiments_to_evaluate, base_eval_params, GT_LABELS_BASE_DIR_HDF5, PROJECT_ROOT are available from Cell 1

if 'tuning_experiments_to_evaluate' not in locals() or not tuning_experiments_to_evaluate:
    print("ERROR: 'tuning_experiments_to_evaluate' not found or empty. Please run Cell 1 first.")
else:
    gt_velocity_thresholds = np.arange(0.1, 3.1, 0.2) # Or your desired range

    # Determine the evaluation mode from base_eval_params (set in Cell 1)
    # This uses your preference for current_eval_params_to_use['evaluate_only_keyframes']
    # by assuming base_eval_params in Cell 1 was set up using that.
    eval_mode_is_keyframes = base_eval_params.get('evaluate_only_keyframes', False)

    print(f"Preparing tasks for {len(tuning_experiments_to_evaluate)} tuning(s) across {len(gt_velocity_thresholds)} GT Velocity Thresholds.")
    print(f"Evaluation mode for plots: {'Keyframes Only' if eval_mode_is_keyframes else 'All Sweeps'}")

    # --- Create all tasks for the multiprocessing pool ---
    tasks_for_plot_data_pool = []
    for experiment_info in tuning_experiments_to_evaluate:
        tuning_name = experiment_info["name"]
        mdet_experiment_dir = experiment_info["path"]
        for threshold in gt_velocity_thresholds:
            # base_eval_params already contains the 'evaluate_only_keyframes' setting
            eval_params_for_task = base_eval_params.copy()
            eval_params_for_task["gt_velocity_threshold"] = threshold
            
            tasks_for_plot_data_pool.append((
                tuning_name,
                mdet_experiment_dir,
                GT_LABELS_BASE_DIR_HDF5, # From Cell 1
                eval_params_for_task,
                PROJECT_ROOT # From Cell 1, for sys.path in worker
            ))

    # --- Dictionary to store raw results from the pool ---
    # Structure: {tuning_name: {threshold: summary_dict, ...}, ...}
    collected_plot_points_from_pool = {exp_info["name"]: {} for exp_info in tuning_experiments_to_evaluate}
    
    if tasks_for_plot_data_pool:
        num_plot_workers = min(multiprocessing.cpu_count(), len(tasks_for_plot_data_pool), 26) # Adjust cap as needed
        print(f"Calculating plot data in parallel using {num_plot_workers} workers for {len(tasks_for_plot_data_pool)} tasks...")

        with multiprocessing.Pool(processes=num_plot_workers) as pool:
            with tqdm(total=len(tasks_for_plot_data_pool), desc="Generating Plot Data") as pbar:
                for tuning_name_res, threshold_res, summary_res in pool.imap_unordered(plot_data_worker, tasks_for_plot_data_pool):
                    if summary_res and not summary_res.get("error"):
                        if tuning_name_res not in collected_plot_points_from_pool: # Should not happen if initialized
                             collected_plot_points_from_pool[tuning_name_res] = {}
                        collected_plot_points_from_pool[tuning_name_res][threshold_res] = summary_res
                    else:
                        # Log or handle failed calculations for a specific point
                        error_detail = summary_res.get("error", "Unknown error") if summary_res else "Worker returned None"
                        # print(f"  Warning: Failed to get summary for {tuning_name_res} at threshold {threshold_res:.2f}. Error: {error_detail}")
                        # Ensure a placeholder if you expect all thresholds for all tunings
                        if tuning_name_res not in collected_plot_points_from_pool:
                             collected_plot_points_from_pool[tuning_name_res] = {}
                        collected_plot_points_from_pool[tuning_name_res][threshold_res] = None # Mark as failed/missing
                    pbar.update(1)
    else:
        print("No tasks generated for plot data calculation.")

    # --- Restructure data for plotting ---
    all_tunings_plot_data_final = {}
    for tuning_name, threshold_data_map in collected_plot_points_from_pool.items():
        current_tuning_metrics_for_plot = {
            "precisions_vs_thresh": [],
            "recalls_vs_thresh": [],
            "f1_scores_vs_thresh": [],
            "ious_dynamic_vs_thresh": []
        }
        for thresh_val in gt_velocity_thresholds: # Iterate in the defined order for consistent plotting
            summary_at_thresh = threshold_data_map.get(thresh_val) # Get the summary for this specific threshold
            if summary_at_thresh and not summary_at_thresh.get("error"):
                current_tuning_metrics_for_plot["precisions_vs_thresh"].append(summary_at_thresh.get('Precision', float('nan')))
                current_tuning_metrics_for_plot["recalls_vs_thresh"].append(summary_at_thresh.get('Recall', float('nan')))
                current_tuning_metrics_for_plot["f1_scores_vs_thresh"].append(summary_at_thresh.get('F1', float('nan')))
                current_tuning_metrics_for_plot["ious_dynamic_vs_thresh"].append(summary_at_thresh.get('overall_iou_dynamic', float('nan')))
            else:
                current_tuning_metrics_for_plot["precisions_vs_thresh"].append(float('nan'))
                current_tuning_metrics_for_plot["recalls_vs_thresh"].append(float('nan'))
                current_tuning_metrics_for_plot["f1_scores_vs_thresh"].append(float('nan'))
                current_tuning_metrics_for_plot["ious_dynamic_vs_thresh"].append(float('nan'))
        all_tunings_plot_data_final[tuning_name] = current_tuning_metrics_for_plot
    
    print("\nFinished calculating all metrics for plotting.\n")

    


In [ ]:
# --- Plotting (using all_tunings_plot_data_final) ---
if not all_tunings_plot_data_final:
    print("No data collected for plotting.")
else:
    num_tunings = len(all_tunings_plot_data_final)
    if num_tunings == 0 :
        print("No tuning data to plot.")
    else:
        if num_tunings <= 10: colors = plt.cm.get_cmap('tab10').colors
        else: colors = plt.cm.get_cmap('viridis')(np.linspace(0, 1, num_tunings))
        linestyles = ['-', '--', '-.', ':'] * (num_tunings // 4 + 1)

        fig, axs = plt.subplots(2, 2, figsize=(18, 14), sharex=True) # Slightly larger figure
        axs = axs.ravel()
        plot_metrics_keys = [
            ("Precision", "precisions_vs_thresh"), ("Recall", "recalls_vs_thresh"),
            ("F1 Score", "f1_scores_vs_thresh"), ("Dynamic IoU", "ious_dynamic_vs_thresh")
        ]

        for i, (metric_label, metric_data_key) in enumerate(plot_metrics_keys):
            ax = axs[i]
            for j, (tuning_name, metrics_data) in enumerate(all_tunings_plot_data_final.items()):
                metric_values = metrics_data[metric_data_key]
                # Plot only if there's valid data to prevent errors with all-NaN slices
                if not all(np.isnan(m) for m in metric_values):
                    ax.plot(gt_velocity_thresholds, metric_values,
                            label=tuning_name, color=colors[j % len(colors)],
                            linestyle=linestyles[j % len(linestyles)],
                            marker='.', markersize=5, alpha=0.8) # Adjusted marker and alpha
            ax.set_xlabel("Ground Truth Velocity Threshold (m/s)")
            ax.set_ylabel(metric_label)
            ax.set_title(f"{metric_label} vs. GT Velocity Threshold")
            ax.grid(True, linestyle='--', alpha=0.6)
            # Adjust legend position for many items
            if num_tunings > 6: # Example threshold
                ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize='small', framealpha=0.7)
            else:
                ax.legend(loc='best', fontsize='small', framealpha=0.8)


        plot_mode_title_suffix = " (Keyframes Only)" if eval_mode_is_keyframes else " (All Sweeps)"
        fig.suptitle(f"M-Detector Performance vs. GT Velocity Threshold{plot_mode_title_suffix}", fontsize=18, y=1.0) # Adjusted y for suptitle
        plt.tight_layout(rect=[0, 0, 0.85 if num_tunings > 6 else 1, 0.96]) # Adjust right margin if legend is outside
        plt.show()

        # --- Summary Table (similar to your existing one) ---
        summary_for_table = []
        for tuning_name, metrics_data in all_tunings_plot_data_final.items():
            f1_scores_list = np.array(metrics_data['f1_scores_vs_thresh'])
            valid_f1_scores = f1_scores_list[~np.isnan(f1_scores_list)]
            if len(valid_f1_scores) > 0:
                max_f1_idx = np.nanargmax(f1_scores_list) # nanargmax correctly handles NaNs
                max_f1 = f1_scores_list[max_f1_idx]
                thresh_at_max_f1 = gt_velocity_thresholds[max_f1_idx]
                precision_at_max_f1 = metrics_data['precisions_vs_thresh'][max_f1_idx]
                recall_at_max_f1 = metrics_data['recalls_vs_thresh'][max_f1_idx]
                summary_for_table.append({
                    "Tuning": tuning_name, "Max F1": max_f1,
                    "GT Vel Thresh @ Max F1": thresh_at_max_f1,
                    "Precision @ Max F1": precision_at_max_f1,
                    "Recall @ Max F1": recall_at_max_f1
                })
            else:
                summary_for_table.append({
                    "Tuning": tuning_name, "Max F1": np.nan, "GT Vel Thresh @ Max F1": np.nan,
                    "Precision @ Max F1": np.nan, "Recall @ Max F1": np.nan
                })
        if summary_for_table:
            summary_df_plot = pd.DataFrame(summary_for_table).set_index("Tuning")
            print(f"\n--- Summary of Max F1 Scores from Plots{plot_mode_title_suffix} ---")
            print(summary_df_plot.to_string(float_format="%.4f"))

print("\nCell 2: Plotting vs. GT Velocity Threshold - OK")